RADIUS sta per **Remote Authentication Dial-In User Service**. Il nome deriva dall'epoca in cui gli utenti si collegavano ad internet tramite modem e linee telefoniche, il suo utilizzo si è poi esteso ad altri tipi di accesso alla rete. 

Il problema da risolvere è la gestione degli utenti in una rete con molti punti di accesso: immaginiamo un operatore che ha molti dispositivi attraverso cui gli utenti possono connettersi --> gestire separatamente su ogni dispositivo password, autorizzazioni e configurazioni è complicato, ogni modifica dovrebbe essere replicata su tutti i dispositivi.

RADIUS permette di **centralizzare le decisioni di accesso**, mantenendo distribuiti i dispositivi che collegano concretamente gli utenti alla rete.  
I partecipanti sono:
- **Utente finale** == chi vuole accedere alla rete, presenta le proprie credenziali al dispositivo di accesso.
- **NAS** (Network Access Server) == gestisce l'accesso dell'utente e applica la decisione del server
- **Server RADIUS** == riceve le richieste di accesso dal NAS, verifica le credenziali e stabilisce quali servizi autorizzare

La cosa importante da capire è che **il client che comunica con il server RADIUS non è l'utente finale, ma il NAS**! A sua volta il NAS comunica con l'utente finale però tramite il protocollo previsto dal sistema di accesso (es. PPP), mentre con il server RADIUS comunica tramite il protocollo RADIUS.

In generale il funzionamento è il seguente:
1. L'utente presenta al NAS le informazioni necessarie per autenticarsi
2. il NAS le inserisce in una richiesta RADIuS e le invia al server RADIUS
3. il server RADIUS verifica le credenziali e risponde al NAS con un messaggio di accettazione o rifiuto
4. il NAS applica la decisione e informa l'utente

RADIUS è un protocollo che fornisce le funzionalità **AAA**: **Authentication** (l'utente è davvero chi dice di essere? es. verifica credenziali), **Authorization** (quali servizi può utilizzare? es. controlla le autorizzazioni) e **Accounting** (quali risorse ha utilizzato? es. durata sessione, traffico, info per la fatturazione).

RADIUS classico è protocollo client server basato su UDP/IP, con porta 1812 per l'autenticazione. La centralizzazione è logica: possono chiaramente esserci server secondari/repliche.

La richiesta NAS prende il nome di **Access-Request**. Le risposte principali sono *Access-Accept* (accesso autorizzato), *Access-Reject* (accesso negato) e Access-Challenge (richiesta di ulteriori informazioni).

Access-Challenge non indica però necessariamente challenge crittografica: infatti alcuni metodi richiedono più messaggi, come quelli che trasportano uno scambio TLS (come vedremo).

In generale un server RADIUS può inoltrare la richiesta ad un altro server ancora, comportandosi quindi come **proxy**.  
Ciò è utile ad es. nel roaming: l'utente accede attraverso la rete ospitante, ma le sue credenziali devono essere verificate dal server del suo provider.  
Per i nostri scopi è sufficiente capire che il NAS può ricevere una decisione ottenuta da più server, concentreremo l'analisi di sicurezza però sul collegamento diretto tra NAS e server RADIUS.

Il server dispone di informazioni relative a **utenti** (credenziali, metodo di autenticazione, autorizzazioni...), **client RADIUS** (NAS autorizzati a comunicare col server), **accounting** (info sull'utilizzo dei servizi)

### Funzionamento RADIUS
Due meccanismi principali per il RADIUS originario:
- **Autenticazione delle risposte del server**, mediante segreto condiviso
- **Protezione della password dell'utente**, mediante meccanismo specifico di RADIUS

Prima di studiarli, distinguiamo i due segreti:
$$P_U = \text{password dell'utente} \quad S = \text{segreto condiviso tra NAS e server RADIUS}$$
La password permette di autenticare l'utente, mentre il segreto $S$ serve a proteggere la comunicazione tra NAS e server RADIUS. **L'utente non deve conoscere il segreto $S$**.

Tra i problemi del protocollo RADIUS originario:
- le risposte del server sono autenticate, mentre dalla parte del NAS (gli Access-Request) no
- l'autenticazione della risposta usa una costruzione basata direttamente su MD5 invece che su HMAC-MD5
- MD5 è fissato dal protocollo, quindi non è possibile sostituirlo con un algoritmo più sicuro
- lo stesso segreto viene impiegato anche per proteggere la password

Nel modello che considereremo si assume che **il NAS e il server RADIUS siano fidati**, che **l'utente possa essere attaccante** e **possa intercettare e modificare i messaggi tra NAS e server RADIUS**.  
Il fatto che NAS sia assunto fidato è necessario perché è il dispositivo che poi applica la decisione --> se fosse compromesso potrebbe consentire l'accesso a chiunque.

La preoccupazione che ci interessa riguarda le risposte del server: se il server risponde con *Access-Reject* e un attaccante riesce a trasformare il messaggio in *Access-Accept* allora è nu burdel.  
Inoltre le risposte del server contengono attributi che specificano le autorizzazioni dell'utente, quindi non basta proteggere il "sì" o il "no", ma anche quali servizi vengono concessi.

### Autenticazione della risposta RADIUS
Siano:
$$I = \text{Identifier della richiesta}, \quad A = \text{Request-Authenticator}$$
Allora il NAS genera $A$ come valore casuale a 16 byte e lo inserisce nell'Access-Request.  
In un Access-Request, $A$ **svolge il ruolo di nonce** (il nome confonde, da solo questo campo non è MAC della richiesta).  
Il NAS conserva localmente le info necessarie ad associare la richiesta alla sua risposta, incluso il valore $A$. 

Quando il server riceve la richiesta, prende la decisione e calcola un tag:
$$t = MD5(\text{Code} || I || \text{Length} || A || \text{Attributes} || S)$$
I campi Code, Identifier, Length e Attributes sono quelli della risposta. $A$ è invece il **Request-Authenticator** della richiesta originale.

Il tag viene inserito nel campo Authenticator della risposta, che in questo caso prende il nome di **Response-Authenticator**. Quindi il campo Authenticator assume due significati nel protocollo RADIUS in base al pacchetto:
- se pacchetto Access-Request --> valore casuale usato come nonce
- se pacchetto Access-Accept/Reject/Challenge --> tag di autenticazione della risposta

Quando il NAS riceve la risposta:
1. individua la richiesta corrispondente e recupera il nonce $A$ conservato
2. ricalcola il tag $t$ usando la risposta ricevuta e il segreto $S$
3. confronta il risultato con il Response-Authenticator

Se i valori coincidono --> verifica riuscita

**Perché serve il segreto?** Per impedire all'attaccante di costruire liberamente un tag valido per una risposta modificata **(autenticazione del messaggio --> protezione da falsificazioni e modifiche i.e. integrità)**
**Perché serve il nonce?** Per collegare la risposta alla richiesta corrente. Senza il nonce l'attaccante potrebbe ripresentare una vecchia risposta positiva **(nonce e gestione dello stato --> protezione da replay)**

**Perché è il NAS a scegliere il nonce?** Perché è il NAS a dover verificare la risposta, quindi è lui che deve poterla associare alla richiesta corrente. Se fosse il server a scegliere il nonce --> il NAS dovrebbe disporre di un ulteriore meccanismo per stabilire se la risposta è nuova. La presenza di molti NAS rende questo meccanismo difficile. 

Il limite di questa protezione è che sì, copre la risposta e il nonce della richiesta, **ma non copre tutti gli attributi della richiesta Access-Request**: questa distinzione sarà il punto dell'attacco a CHAP (una risposta autentica e fresca può essere la risposta a una richiesta che un attaccante ha modificato!)

#### Il formato del pacchetto
Un pacchetto RADIUS ha un'intestazione da 20 byte, seguita da un payload "Attributes" varibaile (max totale 4096 byte):

<img src="imgs/radius_header.png" width="400"/>

Tra i codici troviamo *1 = Access-Request*, *2 = Access-Accept*, *3 = Access-Reject*, *4 = Accounting-Request*, *5 = Accounting-Response*, *11 = Access-Challenge*. Ovviamente qui non serve sapere a memoria i numeri degli attributi, basta sapere come è fatto il pacchetto. 

L'identifier ha solo 8 bit quindi 256 valori possibili, serve ad **associare la risposta alla richiesta**. Da non confondere con il nonce che invece è il Request-Authenticator, è un campo da 128 bit e serve come detto a fornire freschezza crittografica (del resto con 128 bit per il paradosso del compleanno come visto le collisioni diventano probabili dopo circa 2^64 richieste!).

Gli attributi seguono il formato **Type-Length-Value* (TLV):
- Type identifica l'attributo specifico (es. User-Name, User-Password, NAS-Port, Service-Type, CHAP-Challenge... 8 bit e quindi 256 tipi possibili)
- Length ne indica la lunghezza 
- Value contiene il valore vero e proprio dell'attributo

Una richiesta Access-Request in particolare contiene tipicamente l'identità dichiarata dall'utente, il materiale necessario a verificarla, l'identità del NAS, la porta attraverso cui l'utente accede, il servizio richiesto...  
Il materiale di autenticazione dipende dal metodo specifico: con *PAP* si usa **User-Password**, con *CHAP* invece **CHAP-Password** che contiene la risposta CHAP e le informazioni necessarie a verificarla.

#### Come RADIUS protegge la password se si usa PAP
Con PAP l'utente presenta username e password.  
Si assume nel modello presentato che il collegamento tra utente e NAS sia protetto, mentre tra NAS e server RADIUS possano esserci attaccanti. 

Per evitare di inoltrare le password in chiaro tra NAS e server, RADIUS usa un meccanismo basato sempre su MD5. Consideriamo una pass lunga al massimo 16 byte.  
Per prima cosa il NAS aggiunge byte 0 fino ad ottenere il blocco $p1$ da 16 byte. Dopodiché calcola:
$$b_1 = MD5(S || A)$$
dove $S$ è il segreto condiviso e $A$ è il Request-Authenticator della richiesta.

MD5 produce un hash da 16 byte --> il NAS combina in XOR il blocco della password con questo valore:
$$c_1 = p_1 \oplus b_1$$
Il NAS inserisce $c_1$ nel campo User-Password della richiesta Access-Request.  
Allora poichè il server conosce $S$ e riceve $A$, può calcolare $b_1$ e recuperare $p_1 = c_1 \oplus b_1$: ci si ricollega quindi di nuovo all'utilizzo di una **maschera combinata tramite XOR**. 

Se invece la pass era più lunga di 16 byte --> si effettua sempre padding con byte 0 fino a una lunghezza multipla di 16 byte e suddivisa in blocchi $p_1, p_2, ..., p_l$.  
Allora per il primo blocco:
$$b_1 = MD5(S || A) \quad c_1 = p_1 \oplus b_1$$
Per i blocchi successivi:
$$b_i = MD5(S || c_{i-1}) \quad c_i = p_i \oplus b_i$$
In questo modo si utilizza una maschera diversa per ogni blocco (simile a quantio visto con CBC nei cifrari a blocchi).

**Le critiche da fare a RADIUS per questo meccanismo sono:**
- impiego di costruzione crittografica specifica del protocollo invece di un algoritmo standard studiato per la cifratura
- dipendenza da MD5
- riutilizzo dello stesso segreto $S$ per proteggere la password e autenticare le risposte del server
- la dipendenza dalla corretta generazione del Request-Authenticator $A$ 

Le ultime due criticità sono evidenti in quanto se due richieste usano lo stesso $S$ E lo stesso $A$ allora per il primo blocco si ottiene la stessa maschera $b_1 = MD5(S || A)$, e se l'attaccante se ne accorge:
$$c_1 \oplus c_1' = p_1 \oplus b_1 \oplus p_1' \oplus b_1 = p_1 \oplus p_1'$$
a quel punto se conosce una delle due password può ricavare l'altra.

Vedremo in dettaglio gli attacchi più avanti, qui basta capire che il solo fatto di usare un hash non garantisce che una costruzione sia un generatore sicuro di keystream e che le collisioni note di MD5 non dimostrano che ogni costruzione basata su MD5 sia violabile allo stesso modo.

Prima di proseguire con CHAP ricordiamo le possibili risposte del server a una richiesta Access-Request:
- Access-Accept --> accesso autorizzato, può contenere configurazioni come il servizio concesso, l'indirizzo IP e altri parametri
- Access-Reject --> accesso negato, può dipendere da credenziali non valide o richiesta non autorizzata anche se l'identità è stata verificata
- Access-Challenge --> la procedura deve proseguire. In tal caso il NAS raccoglie nuove info dall'utente e invia un nuovo Access-Request.

#### Come RADIUS protegge la password se si usa CHAP
Con CHAP ricordiamo si usa una challenge e una risposta calcolata tramite il segreto dell'utente. Indichiamo:
$$q = \text{challenge} \quad r = \text{risposta} \quad P_u = \text{password dell'utente}$$
La forma semplificata che usiamo per il ragionamento è $r = H(q, P_u)$, ma in realtà nel CHAP-MD5 standard interviene anche un identificatore CHAP $j$ che serve a distinguere più challenge in una stessa sessione (diverso dall'identifier del pacchetto RADIUS!) $r = MD5(j || P_u || q)$.

Procedura:
1. il NAS genera la challenge $q$, e la invia all'utente
2. l'utente calcola la risposta $r$ usando la propria password
3. il NAS inoltra al server username, challenge e risposta CHAP
4. il server recupera il materiale segreto dell'utente, ricalcola la risposta e confronta per poi inviare Access-Accept oppure Access-Reject

La cosa importante è che quindi **il NAS può generare la challenge senza conoscere la password**, ma non può ovviamente verificare la risposta se non dispone del materiale segreto necessario.  
Nel formato RADIUS l'attributo CHAP-Password quindi **non contiene la password dell'utente, ma l'identificatore CHAP e la risposta calcolata**.

Inoltre è essenziale distinguere i ruoli di $q$ e $A$ nel protocollo: la challenge CHAP $q$ permette al server di verificare la risposta dell'utente, mentre il Request-Authenticator $A$ serve a collegare la risposta RADIUS alla richiesta corrente del NAS.  

(NB. Challenge e risposta CHAP possono essere intercettate, ma la password non passa mai nel canale di comunicazione come vuole CHAP e inoltre ovviamente la risposta CHAP è un hash e quindi serve solo per verifica tramite confronto, non per recuperare la password dell'utente! Nun se po' fa'!)

##### Attacco a CHAP con RADIUS
Consideriamo il meccanismo descritto finora, dove **gli attributi dell'Access-Request non sono protetti da MAC** e consideriamo un attaccante capace di modificarli nel canale NAS-server.  
L'attaccante non conosce né la password di Lisa, né il segreto condiviso $S$. Ha però osservato una precedente autenticazione valida di Lisa.
1. **L'attaccante registra uno scambio CHAP valido**: durante l'accesso legittimo, Lisa ha ricevuto una challenge $q_0 = 1234$ e ha prodotto la risposta $r_0 = H(q_0, P_{Lisa}) = \alpha \beta \gamma \delta$. Allora l'attaccante conserva la tupla valida:
   $$(\text{Lisa}, j_0, q_0, r_0)$$
   che soddisfa r_0 = MD5(j_0 || P_{Lisa} || q_0)
2. **L'attaccante avvia un nuovo accesso dichiarandosi Lisa** --> il NAS genera una nuova challenge $q_1 = 5678$.  
L'attaccante non conosce la password quindi non può generare la risposta corretta, allora invia un valore arbitrario $r_{fake}$. Il NAS non può verificarla direttamente --> prepara la richiesta RADIUS contenente:
   $$Q = (I_1, A_1, \text{Lisa}, j_1, q_1, r_{fake}, \text{servizio richiesto})$$
   Il NAS conserva $A_1$ per verificare la successiva risposta RADIUS come visto all'inizio. 
3. Se il server ricevesse questa richiesta invariata, l'autenticazione fallirebbe.  
Ma l'attaccante **modifica la richiesta tra NAS e server**: intercetta $Q$ e sostituisce i dati CHAP con quelli osservati nella vecchia autenticazione valida:
   $$(j_1, q_1, r_{fake}) \to (j_0, q_0, r_0)$$
   Lasciando invariati l'Identifier RADIUS $I_1$ e il Request-Authenticator $A_1$. Il server riceve quindi:
   $$Q' = (I_1, A_1, \text{Lisa}, j_0, q_0, r_0, \text{servizio richiesto})$$
   che non è replay dell'intero pacchetto (non sarebbe valido perché l'Identifier RADIUS è diverso), ma **solo dei campi CHAP**. 
4. **Il server verifica i dati sostituiti e risponde positivamente**: infatti per il server la coppia challenge-risposta è valida: $r_0 = MD5(j_0 || P_{Lisa} || q_0)$, quindi invia Access-Accept al NAS autenticato con $I_1$, $A_1$ e $S$. 
5. **Il NAS accetta una risposta autentica alla richiesta alterata**: riceve l'Access-Accept e verifica il tag usando il nonce $A_1$ che aveva conservato, la verifica riesce.

In questo caso il nonce non impedisce l'attacco perché l'attaccante non ripresenta una vecchia risposta radius, ma ne modifica una nuova intercettata.  
Questo dimostra come in questo caso **la freshness della risposta RADIUS non prova l'integrità della richiesta Access-Request!**

**Perché non basta memorizzare la richiesta sul NAS**? Il NAS potrebbe conservare il fatto di aver inviato $(\text{Lisa}, j_1, q_1, r_{fake})$, ma l'Access-Accept dal server non restituisce necessariamente quei dati in una forma autenticata che permetta di confrontarli con quelli effettivamente verificati da parte del server.  
Il tag della risposta copre solo la risposta e $A_1$, non contiene automaticamente la challenge e la risposta CHAP dell'Access-Request, i.e. il NAS può essere sicuro che il server ha risposto positivamente, ma non che abbia ricevuto gli stessi attributi che lui gli aveva inviato. 

Per eliminare la possibilità dell'attacco ci sono due possibilità:
1. **Proteggere anche l'integrità della richiesta**, di modo che l'attaccante non possa sostituire gli attributi
2. **Legare crittograficamente la risposta del server al contenuto della richiesta**, di modo che il NAS possa verificare che il server abbia elaborato la richiesta originale.



### RADIUS Security Weaknesses e come ovviare ad esse
A conti fatti, RADIUS per come l'abbiamo descritto (protocollo classico) ha due problemi principali:
1. **non cifra l'intero pacchetto**: vari attributi come username, identificazione del NAS e info sulla stazione che accede possono essere leggibili anche da chi intercetta il traffico (proteggere la password non equivale a proteggere tutte le info trasmesse)
2. **autenticazione della risposta, ma non della richiesta Access-Request**


(Message Authenticator e EAP)